# **EDA Notebook**



---
## 0. Setup Environment

In [39]:
# Do not modify this code
%pip install uts_36106_core==0.0.3

from uts_36106_core.folders import *
from uts_36106_core.ipyrenders import *
import os

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.mount_gdrive()
at.create_folders()
if at.is_colab:
  os.chdir(f'{at.folder_path}')

import warnings
warnings.simplefilter(action='ignore')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [40]:
group_name = "Group 52"
student_name = "Andres Ethorimn Lobarb"
student_id = "26029165"

In [41]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [42]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [43]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [44]:
# No additional packages: pandas, numpy, scipy and altair are available on Colab.

### 0.b Import Packages

In [45]:
import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import norm, spearmanr

In [46]:
# Utils

def column_profile(df):
    """One row per column: type, missing count and share, distinct values."""
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_share": df.isna().mean().round(4),
        "distinct": df.nunique(),
    })


def monthly_activity(transactions, timestamp_col, account_col, amount_col, intl_col):
    """Aggregate transactions per account and calendar month."""
    month = pd.to_datetime(transactions[timestamp_col]).dt.to_period("M")
    grouped = transactions.assign(month=month).groupby([account_col, "month"])
    return grouped.agg(
        monthly_spend=(amount_col, "sum"),
        transaction_count=(amount_col, "size"),
        international_share=(intl_col, "mean"),
    ).reset_index()


def forward_change(balances, account_col, date_col, balance_col, horizon):
    """Balance change from each month-end to the month-end `horizon` months later."""
    ordered = balances.sort_values([account_col, date_col])
    future = ordered.groupby(account_col)[balance_col].shift(-horizon)
    out = ordered[[account_col, date_col, balance_col]].copy()
    out["future_change"] = future - ordered[balance_col]
    out["month"] = pd.to_datetime(out[date_col]).dt.to_period("M")
    return out


def spearman_summary(x, y, label):
    """Spearman correlation with a Fisher-z 95% interval and a size label."""
    mask = x.notna() & y.notna()
    rho, p_value = spearmanr(x[mask], y[mask])
    n = int(mask.sum())
    half_width = 1.96 / np.sqrt(n - 3)
    low, high = np.tanh(np.arctanh(rho) - half_width), np.tanh(np.arctanh(rho) + half_width)
    size = abs(rho)
    magnitude = "negligible" if size < 0.1 else "small" if size < 0.3 else "medium" if size < 0.5 else "large"
    return pd.Series({"feature": label, "n": n, "rho": round(rho, 4), "ci_low": round(low, 4),
                      "ci_high": round(high, 4), "p_value": p_value, "magnitude": magnitude})


def group_mean(df, group_col, target):
    """Mean of `target` with a 95% interval per group."""
    table = df.groupby(group_col, observed=True)[target].agg(["mean", "std", "size"]).reset_index()
    table = table.rename(columns={"size": "n"})
    half = 1.96 * table["std"] / np.sqrt(table["n"])
    table["low"], table["high"] = table["mean"] - half, table["mean"] + half
    return table


def binned_mean(df, feature, target, n_bins):
    """group_mean over quantile bins of `feature` (1 = lowest)."""
    bins = pd.qcut(df[feature].rank(method="first"), n_bins, labels=False) + 1
    return group_mean(df.assign(bin=bins), "bin", target)


def interval_chart(table, x_field, x_title, y_title, reference, reference_label):
    """Group means with 95% intervals against a grey, labelled reference line.

    The reference is the overall mean: groups on the line show no effect of the feature."""
    base = alt.Chart(table).encode(x=alt.X(f"{x_field}:O", title=x_title, sort=None))
    intervals = base.mark_rule(color="#898781").encode(
        y=alt.Y("low:Q", title=y_title, scale=alt.Scale(zero=False)), y2="high:Q")
    points = base.mark_point(filled=True, color="#2a78d6").encode(
        y="mean:Q", tooltip=[x_field, alt.Tooltip("mean:Q", format=".0f"), alt.Tooltip("low:Q", format=".0f"),
                             alt.Tooltip("high:Q", format=".0f"), "n"])
    ref = pd.DataFrame({"y": [reference], "label": [reference_label]})
    line = alt.Chart(ref).mark_rule(color="#898781", strokeDash=[4, 4]).encode(y="y:Q")
    text = alt.Chart(ref).mark_text(align="left", dx=4, dy=-7, color="#898781").encode(
        y="y:Q", x=alt.value(0), text="label:N")
    return (intervals + points + line + text).properties(width=520, height=260)


def qq_frame(values):
    """Sample quantiles against normal quantiles (all points up to 500 values, else 199 levels).

    Robust standardisation (median, IQR / 1.349) keeps a few extreme values from flattening the plot;
    for normal data it equals the usual standardisation."""
    values = pd.Series(values).dropna()
    n = len(values)
    levels = (np.arange(1, n + 1) - 0.5) / n if n <= 500 else np.linspace(0.005, 0.995, 199)
    scale = (values.quantile(0.75) - values.quantile(0.25)) / 1.349
    scale = scale if scale > 0 else values.std()  # IQR is 0 when most values are equal
    z = (values - values.median()) / scale
    return pd.DataFrame({"theoretical": norm.ppf(levels), "sample": np.quantile(z, levels)})


def distribution_charts(values, label, n_bins, tail):
    """Histogram (the `tail` share at each end not drawn) beside a normal QQ plot, with shape statistics."""
    values = pd.Series(values).dropna()
    low, high = values.quantile(tail), values.quantile(1 - tail)
    shown = values[(values >= low) & (values <= high)]
    counts, edges = np.histogram(shown, bins=n_bins)
    hist = alt.Chart(pd.DataFrame({"start": edges[:-1], "end": edges[1:], "count": counts})).mark_bar(
        color="#2a78d6").encode(x=alt.X("start:Q", bin="binned", title=label), x2="end:Q",
                                y=alt.Y("count:Q", title="Count")).properties(width=300, height=220)
    diagonal = pd.DataFrame({"theoretical": [-3, 3], "sample": [-3, 3]})
    qq = (alt.Chart(qq_frame(values)).mark_point(color="#2a78d6", size=14).encode(
            x=alt.X("theoretical:Q", title="Normal quantile"), y=alt.Y("sample:Q", title="Robust-standardised sample quantile"))
          + alt.Chart(diagonal).mark_line(color="#898781", strokeDash=[4, 4]).encode(x="theoretical:Q", y="sample:Q")
          ).properties(width=300, height=220)
    print(f"{label}: n = {len(values)}, skewness = {values.skew():.2f}, excess kurtosis = {values.kurt():.2f}, "
          f"histogram omits {len(values) - len(shown)} values outside the {tail:.1%} to {1 - tail:.1%} quantiles")
    return alt.hconcat(hist, qq)

In [47]:
# Constants
# Provisional working horizon (months ahead). It is a working assumption until the balance
# history EDA has examined other horizons; it is not a decision taken here.
HORIZON = 3
N_BINS = 10

---
## B. Data Understanding

**Approach.** For each feature we ask one question: does it tell us how the balance moves next? We draw each finding from the outputs above it and leave the horizon, scope and business use for after all three EDA notebooks.

In [48]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "transactions.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [49]:
print("Rows, columns:", df.shape)
print("Duplicate transaction_id:", df["transaction_id"].duplicated().sum())
print("Fully duplicated rows (ignoring the id):", df.drop(columns="transaction_id").duplicated().sum())
timestamps = pd.to_datetime(df["transaction_timestamp"])
print("First and last timestamp:", timestamps.min(), timestamps.max())
print("Accounts:", df["account_id"].nunique(), "| customers:", df["customer_id"].nunique())

Rows, columns: (1532009, 24)
Duplicate transaction_id: 0
Fully duplicated rows (ignoring the id): 0
First and last timestamp: 2023-01-01 00:00:00 2025-12-30 23:59:00
Accounts: 38562 | customers: 15000


In [50]:
display(column_profile(df))

,type,missing,missing_share,distinct
transaction_id,int64,0,0.0000,1532009
customer_id,int64,0,0.0000,15000
account_id,int64,0,0.0000,38562
card_id,float64,390141,0.2547,33555
merchant_id,int64,0,0.0000,150
transaction_timestamp,object,0,0.0000,979531
amount,float64,0,0.0000,151968
amount_band,object,0,0.0000,5
transaction_type,object,0,0.0000,5
channel,object,0,0.0000,4


In [51]:
# Is there a direction (inflow vs outflow) in the amounts?
amount_by_type = df.groupby("transaction_type")["amount"].describe(percentiles=[0.25, 0.5, 0.75])
print("Amount by transaction type (dollars):")
display(amount_by_type.round(2))
print("Transactions with a zero or negative amount:", (df["amount"] <= 0).sum())

Amount by transaction type (dollars):


,count,mean,std,min,25%,50%,75%,max
transaction_type,,,,,,,,
ATM Withdrawal,306499.0,340.19,372.01,6.15,108.10,224.61,434.06,8987.12
Bill Payment,306029.0,340.09,372.43,7.02,107.62,224.05,434.20,9318.58
Direct Debit,307418.0,340.14,372.22,7.16,107.99,224.35,433.63,7799.19
Purchase,305518.0,339.37,369.84,7.16,108.25,224.78,433.64,7378.10
Transfer,306545.0,339.39,372.26,3.60,107.46,224.21,432.56,6881.69


Transactions with a zero or negative amount: 0


In [52]:
print("Transaction type by channel (counts):")
display(pd.crosstab(df["transaction_type"], df["channel"]))

Transaction type by channel (counts):


channel,ATM,Mobile,POS,Web
transaction_type,,,,
ATM Withdrawal,15204,61231,183969,46095
Bill Payment,15382,61004,183668,45975
Direct Debit,15448,61526,183933,46511
Purchase,15206,61072,183564,45676
Transfer,15407,61299,183928,45911


In [53]:
monthly_counts = (timestamps.dt.to_period("M").astype(str).value_counts().sort_index()
                  .rename_axis("month").reset_index(name="transactions"))
alt.Chart(monthly_counts).mark_bar(color="#2a78d6").encode(
    x=alt.X("month:O", title="Month"),
    y=alt.Y("transactions:Q", title="Transactions"),
    tooltip=["month", "transactions"],
).properties(width=620, height=240)

alt.Chart(...)

In [54]:
dataset_insights = """
The table holds 1532009 transactions in 24 columns, covering 38562 accounts of 15000 customers from
2023-01-01 to 2025-12-30, and every month carries a steady 39000 to 44000 transactions. It is clean:
there are no duplicates, and the only missing field is <card_id> (25.5%).

Every amount is positive, Transfers included, so the table records how active an account is but not
whether money comes in or goes out, and a net flow cannot be computed from it. The five types also
carry almost the same amounts (median about $224) and appear through every channel in the same
proportions; 183969 ATM withdrawals, for example, are booked through POS. We therefore leave type
and channel aside and test the three features the table can support: monthly spend, transaction
count and international share.
"""

In [55]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest `monthly_spend`

In [56]:
# Complementary load: balances are needed to measure the change that follows each month.
balances = pd.read_csv(at.folder_path / "account_balance_history.csv",
                       usecols=["account_id", "balance_date", "balance"])
changes = forward_change(balances, "account_id", "balance_date", "balance", HORIZON)

In [57]:
activity = monthly_activity(df, "transaction_timestamp", "account_id", "amount", "international_transaction_flag")
first_month = activity["month"].min()

# One row per account and month-end from the first transaction month; months with no
# transactions get zero spend and zero count (no transaction recorded that month).
panel = changes[changes["month"] >= first_month].merge(activity, on=["account_id", "month"], how="left")
panel[["monthly_spend", "transaction_count"]] = panel[["monthly_spend", "transaction_count"]].fillna(0)
panel = panel.dropna(subset=["future_change"])

distribution_charts(panel["monthly_spend"], "Monthly spend ($)", 40, 0.005)

Monthly spend ($): n = 1272546, skewness = 2.62, excess kurtosis = 10.61, histogram omits 6363 values outside the 0.5% to 99.5% quantiles


alt.HConcatChart(...)

In [58]:
print("Account-months with an observed future change:", len(panel))
print("Share of account-months with no transaction:", round((panel["transaction_count"] == 0).mean(), 4))
display(panel["monthly_spend"].describe().round(2).to_frame("monthly spend (dollars)"))

Account-months with an observed future change: 1272546
Share of account-months with no transaction: 0.3788


,monthly spend (dollars)
count,1272546.00
mean,374.99
std,557.03
min,0.00
25%,0.00
50%,150.15
75%,539.40
max,9588.01


In [59]:
spend_test = spearman_summary(panel["monthly_spend"], panel["future_change"], "monthly spend")
display(spend_test.to_frame("Spearman, monthly spend vs balance change over the HORIZON"))

,"Spearman, monthly spend vs balance change over the HORIZON"
feature,monthly spend
n,1272546
rho,0.0016
ci_low,-0.0001
ci_high,0.0034
p_value,0.063482
magnitude,negligible


In [60]:
spend_bins = binned_mean(panel, "monthly_spend", "future_change", N_BINS)
overall = panel["future_change"].mean()
interval_chart(
    spend_bins, "bin", "Monthly spend decile (1 = lowest)", f"Mean balance change over the next {HORIZON} months ($)",
    overall, f"all account-months: ${overall:.0f}"
)

alt.LayerChart(...)

In [61]:
feature_1_insights = """
For each account-month we compared monthly spend with how the balance moved over the following
<HORIZON> months (3, provisional), across 1272546 account-months. Spend is strongly right-skewed
(median $150, mean $375), and 37.9% of these months have no transaction at all.

The association is negligible (Spearman rho = 0.0016, 95% interval -0.0001 to 0.0034): whatever
customers spend, their balance moves on average by about $640 to $720, close to the overall $686
(chart), so spend does not tell the bank where a balance is heading.
"""

In [62]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `transaction_count`

In [63]:
distribution_charts(panel["transaction_count"], "Transactions in the month", 12, 0)

Transactions in the month: n = 1272546, skewness = 1.19, excess kurtosis = 1.57, histogram omits 0 values outside the 0.0% to 100.0% quantiles


alt.HConcatChart(...)

In [64]:
display(panel["transaction_count"].value_counts().sort_index().to_frame("account-months"))
count_test = spearman_summary(panel["transaction_count"], panel["future_change"], "transaction count")
display(count_test.to_frame("Spearman, transactions in the month vs balance change over the next HORIZON months"))

,account-months
transaction_count,
0.0,482064
1.0,405407
2.0,230274
3.0,101926
4.0,36948
5.0,11633
6.0,3250
7.0,803
8.0,183


,"Spearman, transactions in the month vs balance change over the next HORIZON months"
feature,transaction count
n,1272546
rho,0.0012
ci_low,-0.0006
ci_high,0.0029
p_value,0.185688
magnitude,negligible


In [65]:
count_table = group_mean(panel.assign(transactions=panel["transaction_count"].clip(upper=5)),
                         "transactions", "future_change")
interval_chart(count_table, "transactions", "Transactions in the month (5 = five or more)", f"Mean balance change over the next {HORIZON} months ($)",
               overall, f"all account-months: ${overall:.0f}")

alt.LayerChart(...)

In [66]:
feature_2_insights = """
Most accounts make no transaction or a single one in a month, and a few reach 11.

How active an account is makes no difference to its next move (Spearman rho = 0.0012, 95% interval
-0.0006 to 0.0029). The average change stays between about $665 and $745 at every level of activity,
and the wider intervals at 4 and at 5 or more transactions reflect fewer account-months rather than
a trend (chart).
"""

In [67]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `international_share`

In [68]:
# Only months with at least one transaction have an international share.
with_activity = panel[panel["transaction_count"] > 0]

distribution_charts(with_activity["international_share"], "International share of the month's transactions", 20, 0)

International share of the month's transactions: n = 790482, skewness = 2.92, excess kurtosis = 7.66, histogram omits 0 values outside the 0.0% to 100.0% quantiles


alt.HConcatChart(...)

In [69]:
print("Account-months with at least one transaction:", len(with_activity))
display(with_activity["international_share"].value_counts(bins=[-0.01, 0, 0.25, 0.5, 0.75, 1]).sort_index()
        .to_frame("account-months"))
intl_test = spearman_summary(with_activity["international_share"], with_activity["future_change"], "international share")
display(intl_test.to_frame("Spearman, international share vs balance change over the next HORIZON months"))

Account-months with at least one transaction: 790482


,account-months
"(-0.011, 0.0]",675507
"(0.0, 0.25]",14605
"(0.25, 0.5]",61100
"(0.5, 0.75]",2294
"(0.75, 1.0]",36976


,"Spearman, international share vs balance change over the next HORIZON months"
feature,international share
n,790482
rho,0.0001
ci_low,-0.0021
ci_high,0.0023
p_value,0.942627
magnitude,negligible


In [70]:
intl_table = group_mean(with_activity.assign(share=pd.cut(with_activity["international_share"], [-0.01, 0, 0.5, 1],
                                                       labels=["none", "up to half", "more than half"])),
                        "share", "future_change")
active_overall = with_activity["future_change"].mean()
interval_chart(intl_table, "share", "Share of the month's transactions that are international", f"Mean balance change over the next {HORIZON} months ($)",
               active_overall, f"all active account-months: ${active_overall:.0f}")

alt.LayerChart(...)

In [71]:
feature_n_insights = """
International spending is rare: 675507 of the 790482 active account-months have none, and otherwise
the share takes only a few values, such as 0.5 or 1, because accounts make few transactions.

Accounts that spend abroad move just like the rest (Spearman rho = 0.0001, 95% interval -0.0021 to
0.0023), at about $690 in every group (chart). None of the three transaction features, in other
words, tells us how a balance will move next (summary table above).
"""

In [72]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

In [73]:
# Summary of the three feature tests in this notebook
display(pd.DataFrame([spend_test, count_test, intl_test]).set_index("feature"))

,n,rho,ci_low,ci_high,p_value,magnitude
feature,,,,,,
monthly spend,1272546,0.0016,-0.0001,0.0034,0.063482,negligible
transaction count,1272546,0.0012,-0.0006,0.0029,0.185688,negligible
international share,790482,0.0001,-0.0021,0.0023,0.942627,negligible


## C. Ethical Consideration of Features


In [74]:
# Fields that can act as proxies for protected or sensitive traits
sensitive = ["merchant_category", "merchant_country", "currency_code", "merchant_state"]
for column in sensitive:
    print(column, "->", df[column].value_counts(normalize=True).round(3).head(6).to_dict())

merchant_category -> {'Groceries': 0.227, 'Retail': 0.173, 'Restaurants': 0.133, 'Entertainment': 0.127, 'Fuel': 0.107, 'Healthcare': 0.08}
merchant_country -> {'Australia': 0.913, 'United Kingdom': 0.04, 'Singapore': 0.02, 'United States': 0.02, 'New Zealand': 0.007}
currency_code -> {'AUD': 0.913, 'GBP': 0.04, 'SGD': 0.02, 'USD': 0.02, 'NZD': 0.007}
merchant_state -> {'SA': 0.173, 'NSW': 0.167, 'QLD': 0.16, 'NT': 0.146, 'ACT': 0.106, 'WA': 0.094}


In [75]:
ethical_considerations = """
Several fields say more about a customer's life than about their banking. Merchant category reveals
health spending (8.0% of transactions are Healthcare), merchant country and currency hint at origin
(8.7% are outside Australia), and state shows where people live. We carry forward only account-level
monthly summaries, and only if later evidence supports them; any use is audited by customer group
and leads to a staff review, never to an automatic decision.
"""

In [76]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='ethical_considerations', value=ethical_considerations)